# SemCred-FL Colab fail-safe

This notebook restores and validates the portable project bundle when local execution must move to Colab. It covers environment and test gates, the bounded Phase 4 check, the Phase 5 synthetic gate, Phase 6B.2 calibration recovery, and resumable Phase 6C attack execution. All real-data runs are disabled by default.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Open this notebook in Google Colab.') from exc

drive.mount('/content/drive')
print('Drive mounted.')

## Configuration

Edit the two Drive paths if your folder name differs. Keep the extraction directory under `/content` for faster local Colab I/O.

In [ ]:
BUNDLE_PATH = Path('/content/drive/MyDrive/SemCred-FL/semcred_fl_colab_bundle.zip')
PROJECT_ROOT = Path('/content/SemCred-FL')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/SemCred-FL/results')

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Bundle not found: {BUNDLE_PATH}')
print(f'Bundle: {BUNDLE_PATH}')
print(f'Extract to: {PROJECT_ROOT}')

## Verify and extract

Every archived file is checked against the internal SHA-256 manifest before setup continues. Existing extraction content is removed only from the explicit `/content/SemCred-FL` directory.

In [ ]:
import hashlib
import json
import shutil
import zipfile
from pathlib import PurePosixPath

MANIFEST_NAME = 'COLAB_BUNDLE_MANIFEST.json'
CHUNK_SIZE = 1024 * 1024

def safe_relative(name: str) -> Path:
    posix = PurePosixPath(name)
    if posix.is_absolute() or '..' in posix.parts or not posix.parts:
        raise ValueError(f'Unsafe archive path: {name}')
    return Path(*posix.parts)

def stream_sha256(stream) -> str:
    digest = hashlib.sha256()
    while chunk := stream.read(CHUNK_SIZE):
        digest.update(chunk)
    return digest.hexdigest()

if PROJECT_ROOT.exists():
    if PROJECT_ROOT.resolve() != Path('/content/SemCred-FL'):
        raise RuntimeError(f'Refusing to replace unexpected path: {PROJECT_ROOT}')
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True)

with zipfile.ZipFile(BUNDLE_PATH, 'r', allowZip64=True) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise ValueError('Duplicate paths in bundle')
    if MANIFEST_NAME not in names:
        raise ValueError('Bundle manifest is missing')
    manifest = json.loads(archive.read(MANIFEST_NAME))
    expected = {MANIFEST_NAME}
    for index, item in enumerate(manifest['files'], start=1):
        name = item['path']
        relative = safe_relative(name)
        expected.add(name)
        info = archive.getinfo(name)
        if info.file_size != item['size']:
            raise ValueError(f'Size mismatch: {name}')
        with archive.open(name) as source:
            digest = stream_sha256(source)
        if digest != item['sha256']:
            raise ValueError(f'Checksum mismatch: {name}')
        target = PROJECT_ROOT / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(name) as source, target.open('wb') as destination:
            shutil.copyfileobj(source, destination, CHUNK_SIZE)
        if index % 25 == 0 or index == len(manifest['files']):
            print(f'verified_and_extracted={index}/{len(manifest["files"])}')
    if set(names) != expected:
        raise ValueError('Bundle contains unmanifested files')

print(f'Bundle ID: {manifest["bundle_id"]}')
print(f'Files: {manifest["file_count"]}')
print(f'Raw datasets included: {manifest["raw_datasets_included"]}')

## Install the Colab environment

Colab's CUDA-enabled PyTorch is retained. The remaining dependencies and the local SemCred-FL package are installed from the verified bundle.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=PROJECT_ROOT,
    check=True,
)
print('Dependencies installed.')

## Mandatory validation gates

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/validate_environment.py'],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, 'scripts/validate_dataset_contracts.py'],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
RUN_FULL_TESTS = True

if RUN_FULL_TESTS:
    subprocess.run(
        [sys.executable, '-m', 'pytest', '-q', '--basetemp=.pytest_tmp_colab'],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Phase 5 synthetic federated-engine gate

This is a resumable implementation check, not a detector-performance experiment.

In [ ]:
RUN_PHASE5_SYNTHETIC = True

if RUN_PHASE5_SYNTHETIC:
    subprocess.run(
        [
            sys.executable,
            'scripts/run_phase5_synthetic.py',
            '--output-directory',
            'artifacts/federation/phase_5/colab_recheck',
            '--rounds',
            '3',
            '--clients-per-round',
            '2',
            '--aggregation',
            'fedavg',
            '--device',
            'cuda',
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Optional bounded Phase 4 GPU check

Set `RUN_PHASE4_SMOKE` to `True` only when you want to recheck one prepared dataset. This bounded run excludes the test partition and cannot support performance claims.

In [ ]:
RUN_PHASE4_SMOKE = False
PHASE4_DATASET = 'edge_iiotset'  # or 'cic_iot_2023'

phase4_inputs = {
    'edge_iiotset': (
        'artifacts/preprocessing/phase_3c/edge_iiotset/edge_iiotset_phase3c_preprocessed.json',
        'artifacts/client_partitions/phase_3d/edge_iiotset/edge_iiotset_phase3d_clients.json',
    ),
    'cic_iot_2023': (
        'artifacts/preprocessing/phase_3c/cic_iot_2023/cic_iot_2023_phase3c_preprocessed.json',
        'artifacts/client_partitions/phase_3d/cic_iot_2023/cic_iot_2023_phase3d_clients.json',
    ),
}

if RUN_PHASE4_SMOKE:
    preprocessing_report, client_manifest = phase4_inputs[PHASE4_DATASET]
    subprocess.run(
        [
            sys.executable,
            'scripts/run_phase4_centralized.py',
            '--preprocessing-report',
            preprocessing_report,
            '--client-partition-manifest',
            client_manifest,
            '--output-directory',
            f'artifacts/centralized/phase_4/colab_recheck/{PHASE4_DATASET}',
            '--epochs',
            '1',
            '--batch-size',
            '256',
            '--train-record-limit',
            '4096',
            '--validation-record-limit',
            '2048',
            '--device',
            'cuda',
            '--progress-every-batches',
            '10',
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Optional Phase 6B.2 benign norm calibration

Run one dataset/profile at a time. The client cache resumes by source shard and federated training resumes at completed round boundaries. An interruption inside a round repeats that round. This is a train-only calibration run, not a performance experiment.

In [ ]:
RUN_PHASE6B_NORM_CALIBRATION = False
PHASE6B_DATASET = 'edge_iiotset'  # or 'cic_iot_2023'
PHASE6B_PROFILE = 'iid'  # iid, dirichlet_0_1, dirichlet_0_3, dirichlet_1_0

if RUN_PHASE6B_NORM_CALIBRATION:
    preprocessing_report, client_manifest = phase4_inputs[PHASE6B_DATASET]
    norm_root = Path('artifacts/attacks/phase_6b/norm_references') / PHASE6B_DATASET / PHASE6B_PROFILE
    cache_root = Path('artifacts/client_cache/phase_6b') / PHASE6B_DATASET / PHASE6B_PROFILE
    subprocess.run(
        [
            sys.executable,
            'scripts/run_phase6b_norm_calibration.py',
            '--preprocessing-report', preprocessing_report,
            '--client-partition-manifest', client_manifest,
            '--profile', PHASE6B_PROFILE,
            '--output-directory', str(norm_root),
            '--cache-directory', str(cache_root),
            '--cache-work-database', str(cache_root / 'client_cache.sqlite3'),
            '--device', 'cuda',
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Optional Phase 6C attack execution

This cell runs or resumes one train-only attack condition. It is disabled by default. Keep `PHASE6C_MAX_ROUNDS_THIS_CALL = 1` for a bounded gate; rerunning advances one completed round. Set it to `None` only when you intend to finish the condition. Adaptive norm additionally loads the matching frozen Phase 6B.2 norm reference.

In [ ]:
RUN_PHASE6C_ATTACK = False
PHASE6C_DATASET = 'edge_iiotset'
PHASE6C_PROFILE = 'iid'
PHASE6C_ATTACK = 'label_flip'
PHASE6C_MALICIOUS_RATIO = 0.1
PHASE6C_SEED = 20260816
PHASE6C_POISON_FRACTION = 0.1
PHASE6C_STRENGTH = None  # clean_label: 0.5 or 1.0
PHASE6C_SCALE = None  # scaling: 5 or 10; sign_reversal: 1, 5, or 10
PHASE6C_MAX_ROUNDS_THIS_CALL = 1

if RUN_PHASE6C_ATTACK:
    preprocessing_report, client_manifest = phase4_inputs[PHASE6C_DATASET]
    resolution = Path('artifacts/attacks/phase_6b') / PHASE6C_DATASET / f'{PHASE6C_DATASET}_phase6b_resolution.json'
    condition = f'{PHASE6C_ATTACK}_mr{PHASE6C_MALICIOUS_RATIO:g}_pf{PHASE6C_POISON_FRACTION:g}_seed{PHASE6C_SEED}'
    output_root = Path('artifacts/attacks/phase_6c') / PHASE6C_DATASET / PHASE6C_PROFILE / condition
    cache_root = Path('artifacts/client_cache/phase_6c') / PHASE6C_DATASET / PHASE6C_PROFILE
    command = [
        sys.executable, 'scripts/run_phase6c_attack.py',
        '--preprocessing-report', preprocessing_report,
        '--client-partition-manifest', client_manifest,
        '--resolution', str(resolution),
        '--dataset', PHASE6C_DATASET,
        '--profile', PHASE6C_PROFILE,
        '--attack-name', PHASE6C_ATTACK,
        '--malicious-ratio', str(PHASE6C_MALICIOUS_RATIO),
        '--seed', str(PHASE6C_SEED),
        '--poison-fraction', str(PHASE6C_POISON_FRACTION),
        '--output-directory', str(output_root),
        '--cache-directory', str(cache_root),
        '--cache-work-database', str(cache_root / 'client_cache.sqlite3'),
        '--device', 'cuda',
    ]
    if PHASE6C_STRENGTH is not None:
        command.extend(['--strength', str(PHASE6C_STRENGTH)])
    if PHASE6C_SCALE is not None:
        command.extend(['--scale', str(PHASE6C_SCALE)])
    if PHASE6C_ATTACK == 'adaptive_norm':
        norm_reference = Path('artifacts/attacks/phase_6b/norm_references') / PHASE6C_DATASET / PHASE6C_PROFILE / 'benign_norm_reference.json'
        command.extend(['--norm-reference', str(norm_reference)])
    if PHASE6C_MAX_ROUNDS_THIS_CALL is not None:
        command.extend(['--max-rounds-this-call', str(PHASE6C_MAX_ROUNDS_THIS_CALL)])
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## Optional Phase 6C sequential matrix

This disabled cell runs the clean and zero-poison controls plus every attack family sequentially to one absolute round target. Completed conditions at that target are skipped. Keep the target at 1 for the implementation gate.

In [ ]:
RUN_PHASE6C_MATRIX = False
PHASE6C_MATRIX_DATASET = 'edge_iiotset'
PHASE6C_MATRIX_PROFILE = 'iid'
PHASE6C_MATRIX_RATIO = 0.1
PHASE6C_MATRIX_SEED = 20260816
PHASE6C_MATRIX_MODE = 'all_types'  # or 'full_grid'
PHASE6C_MATRIX_TARGET_ROUND = 1

if RUN_PHASE6C_MATRIX:
    preprocessing_report, client_manifest = phase4_inputs[PHASE6C_MATRIX_DATASET]
    resolution = Path('artifacts/attacks/phase_6b') / PHASE6C_MATRIX_DATASET / f'{PHASE6C_MATRIX_DATASET}_phase6b_resolution.json'
    subprocess.run(
        [
            sys.executable, 'scripts/run_phase6c_matrix.py',
            '--preprocessing-report', preprocessing_report,
            '--client-partition-manifest', client_manifest,
            '--resolution', str(resolution),
            '--output-root', 'artifacts/attacks/phase_6c',
            '--cache-root', 'artifacts/client_cache/phase_6c',
            '--norm-reference-root', 'artifacts/attacks/phase_6b/norm_references',
            '--dataset', PHASE6C_MATRIX_DATASET,
            '--profiles', PHASE6C_MATRIX_PROFILE,
            '--malicious-ratios', str(PHASE6C_MATRIX_RATIO),
            '--seeds', str(PHASE6C_MATRIX_SEED),
            '--matrix', PHASE6C_MATRIX_MODE,
            '--target-round', str(PHASE6C_MATRIX_TARGET_ROUND),
            '--device', 'cuda',
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Save generated gates to Drive

Only newly generated recheck directories are copied. Prepared Phase 3C/3D inputs remain in the original bundle.

In [ ]:
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
destination = DRIVE_OUTPUT_ROOT / run_stamp
destination.mkdir(parents=True, exist_ok=False)

result_paths = [
    Path('artifacts/federation/phase_5/colab_recheck'),
    Path('artifacts/centralized/phase_4/colab_recheck'),
    Path('artifacts/attacks/phase_6b/norm_references'),
    Path('artifacts/attacks/phase_6c'),
]
for relative in result_paths:
    source = PROJECT_ROOT / relative
    if source.exists():
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, target)

(destination / 'runtime_bundle_id.txt').write_text(
    manifest['bundle_id'] + '\n', encoding='utf-8'
)
print(f'Results copied to: {destination}')

## Stop boundary

Stop here after the selected bounded or resumed Phase 6C condition. The output is train-only attack-generation evidence and cannot support detector-performance claims. Semantic credibility, defense execution, test evaluation, and the confirmatory experiment matrix require later phases. Do not invent commands beyond the checked-in wrappers.